# 90 — Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

Students run `01_load_explore.ipynb`, which pulls live from NOAA. That is the right experience:
they see provenance, and the cleaning decisions are the teaching.

But a live-source notebook has one failure mode we cannot accept — if NOAA's service is slow or
down on event day, every data lane in the room stalls at once with no way forward. So we publish
a snapshot, and `scripts/load.sh` rebuilds from it.

## The important design decision

**This notebook does not reimplement the transformations.** It downloads the student notebook,
runs it once per city with `CITY` overridden, and exports whatever tables that produces.

That matters. If we maintained a second copy of the cleaning logic, the two would drift, and the
fallback would quietly stop matching what students get. Here the snapshot is *by construction*
exactly the output of the notebook we ship. Change the notebook, re-run this, and the fallback
follows.

## Bucket layout

```
gs://class-demo/a4i-2026/<challenge>/<city>/<table>/*.parquet
```

`a4i-2026` is the generic top level so every challenge shares it. Each challenge gets a folder,
each city a subfolder under that, each table a folder of Parquet parts.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET_ROOT = "gs://class-demo/a4i-2026"
CHALLENGE   = "challenge-1-walkability"
DATASET     = "a4i_climate"
LOCATION    = "US"

# The student notebook, fetched live so this cannot drift from what ships.
# Update the org/repo once the template repository is published.
NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/ROI-Training/"
    "A4I2026-challenge-1-walkability/main/notebooks/01_load_explore.ipynb"
)

# Cities to publish. Keep this in sync with VERIFIED_CITIES in the student
# notebook, and add the four relay cities as soon as they are confirmed.
CITIES = [
    "Chicago",
    "Atlanta",
    "Houston",
    "Seattle",
]

TABLES = ["heat_observations", "station_daily_temps", "tract_demographics"]

import google.auth
from google.cloud import bigquery

_, PROJECT_ID = google.auth.default()
client = bigquery.Client(project=PROJECT_ID)

print(f"Project : {PROJECT_ID}")
print(f"Target  : {BUCKET_ROOT}/{CHALLENGE}/<city>/")
print(f"Cities  : {', '.join(CITIES)}")

## Fetch the student notebook and extract its code

We skip `%%bigquery` magic cells — they are display-only exploration and cannot be executed as
plain Python. Everything that builds a table is an ordinary code cell.

In [ ]:
import re, requests, nbformat

resp = requests.get(NOTEBOOK_URL, timeout=60)
resp.raise_for_status()
student_nb = nbformat.reads(resp.text, as_version=4)

STUDENT_CODE = [
    c.source for c in student_nb.cells
    if c.cell_type == "code" and not c.source.lstrip().startswith("%%")
]

print(f"Fetched {len(student_nb.cells)} cells, {len(STUDENT_CODE)} executable.")

## Run the notebook once per city

Each city gets a fresh namespace. The student notebook's own validation section raises on a
failed check, so a city that produces bad tables will stop here rather than being published.
That is deliberate — **we would rather publish nothing for a city than publish something wrong.**

In [ ]:
import traceback

def slugify(city):
    return re.sub(r"[^a-z0-9]+", "-", city.lower()).strip("-")


def build_city(city):
    '''Execute the student notebook with CITY overridden. Returns True on success.'''
    ns = {"__name__": "__main__"}
    for src in STUDENT_CODE:
        src = re.sub(r'^CITY\s*=\s*["\'].*?["\']',
                     f'CITY = "{city}"', src, count=1, flags=re.M)
        exec(compile(src, f"<{slugify(city)}>", "exec"), ns)
    return ns


results = {}
for city in CITIES:
    print("=" * 60)
    print(f"BUILDING: {city}")
    print("=" * 60)
    try:
        build_city(city)
        results[city] = "built"
        print(f"\n--> {city}: tables built and validated\n")
    except Exception as exc:
        results[city] = f"FAILED: {exc}"
        print(f"\n--> {city} FAILED: {exc}")
        traceback.print_exc()
        print()

## Export to Cloud Storage

`EXPORT DATA` writes Parquet server-side — no data passes through this runtime, so it is fast
regardless of table size. The `*.parquet` wildcard lets BigQuery shard if it wants to;
`scripts/load.sh` loads the whole folder, so sharding is invisible to students.

Note this exports whatever is currently in the dataset, so it must run immediately after the
city that produced it. That is why the build and export loops are separate cells but the same
city order.

In [ ]:
def export_city(city):
    slug = slugify(city)
    base = f"{BUCKET_ROOT}/{CHALLENGE}/{slug}"
    for table in TABLES:
        uri = f"{base}/{table}/*.parquet"
        client.query(f'''
            EXPORT DATA OPTIONS(
              uri = '{uri}',
              format = 'PARQUET',
              overwrite = true
            ) AS
            SELECT * FROM `{PROJECT_ID}.{DATASET}.{table}`
        ''').result()
        print(f"   {table:<22} -> {uri}")
    return base


# Rebuild + export, city by city, so each export captures its own tables.
published = {}
for city in CITIES:
    if not results.get(city, "").startswith("built"):
        print(f"SKIPPING {city} - build did not succeed")
        continue
    print(f"\nExporting {city}...")
    build_city(city)          # re-materialise this city's tables
    published[city] = export_city(city)

print("\nPublished:")
for city, path in published.items():
    print(f"  {city:<16} {path}")

## Verify the snapshot is loadable

Publishing something nobody checked is how you find out at 9am on event day. This reads each
exported folder back and reports row counts, using the same wildcard `load.sh` will use.

In [ ]:
ok = True
for city, base in published.items():
    print(f"\n{city}")
    for table in TABLES:
        uri = f"{base}/{table}/*.parquet"
        try:
            # Read straight from GCS via an external table definition,
            # using the same wildcard load.sh will use.
            cfg = bigquery.ExternalConfig("PARQUET")
            cfg.source_uris = [uri]
            job_cfg = bigquery.QueryJobConfig(
                table_definitions={"snap": cfg})
            rows = client.query(
                "SELECT COUNT(*) AS n FROM snap", job_config=job_cfg
            ).to_dataframe().iloc[0]["n"]
            flag = "" if rows > 0 else "   <-- EMPTY"
            print(f"   {table:<22} {int(rows):>10,} rows{flag}")
            if rows == 0:
                ok = False
        except Exception as exc:
            print(f"   {table:<22} UNREADABLE: {exc}")
            ok = False

print("\n" + ("Snapshot verified. scripts/load.sh will work."
              if ok else "PROBLEMS FOUND - do not rely on this snapshot."))

## Make it publicly readable

`scripts/load.sh` runs in a student's own project with their own credentials, so the snapshot
must be readable anonymously.

Run this once per bucket, not per publish. It is here for reference rather than convenience —
check with whoever owns the bucket before changing its IAM.

```bash
gsutil iam ch allUsers:objectViewer gs://class-demo
```

Then confirm from a machine with no credentials:

```bash
curl -sI https://storage.googleapis.com/class-demo/a4i-2026/challenge-1-walkability/chicago/heat_observations/000000000000.parquet | head -1
```

A `200` means students can reach it. A `403` means they cannot, and `load.sh` will fail for
everyone.